# Example 8 — PEPSI legacy line-window validation

## What this example teaches

This example reads bundled PEPSI `.nor` spectra, turns the line windows into a `SpectrumCollection`, and can run the maintained PHOENIX regression script to produce a compact per-line data/model diagnostic similar to Figure 4 of [arXiv:2503.22331](https://arxiv.org/pdf/2503.22331).

## Requirements

The inspection cells use only bundled PEPSI spectra. The model-overlaid validation plot requires a configured local PHOENIX library and is run through `scripts/pepsi_fit_smoketest.py` so the optimizer is maintained in one place.

## Expected outputs

You should see the PEPSI reader assumptions, a prepared window summary, a compact observed-window plot, and, if `RUN_LEGACY_FIT = True`, a six-panel model/data diagnostic with one panel per legacy line window. This is a compatibility/regression workflow, not a generic final-analysis recipe for every PEPSI product.


## 0. Imports and controls

The PEPSI reader name describes the data product. The `.dxt.nor` suffix alone is not enough to infer wavelength frame for every PEPSI release, so the working wavelength hypothesis is kept explicit.


In [ ]:
from pathlib import Path
import subprocess
import sys

from IPython.display import Image, display
import matplotlib.pyplot as plt
import Spyctres as sp

# Find the repository root defensively. This keeps the notebook working when
# Jupyter is launched from the repository root, from examples/, or from an
# editable Spyctres checkout.
root_candidates = []

for start in (Path.cwd().resolve(), Path(sp.__file__).resolve()):
    root_candidates.extend([start, *start.parents])
ROOT = next(
    (
        candidate
        for candidate in root_candidates
        if (candidate / "scripts" / "pepsi_fit_smoketest.py").exists()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate scripts/pepsi_fit_smoketest.py. Run this notebook "
        "from the Spyctres source checkout, or use the matching example script."
    )

pepsi_paths = [
    sp.example_data_path("pepsir.20230603.009.dxt.nor"),
    sp.example_data_path("pepsir.20230603.010.dxt.nor"),
]

reader = "pepsi_nor"
wave_hypothesis = "air"
legacy_halfwidth_A = 10.0
window_pad_A = 2.0

# Keep this True when PHOENIX is configured and you want the Figure-4-style
# model overlay. Set it False for a PHOENIX-free inspection pass.
RUN_LEGACY_FIT = True
legacy_fit_plot = Path("/tmp/spyctres_example8_pepsi_legacy_fit.png")



## 1. Read the PEPSI spectra

`sp.read_spectrum(..., reader="pepsi_nor")` returns the same canonical spectrum container used by the rest of Spyctres. The reader records metadata such as wavelength medium, observer frame, stellar-rest status, and PEPSI velocity keywords when present. It does not silently apply `SSBVEL` or infer a stellar-rest correction from the filename.


In [ ]:
print("Available readers include:", ", ".join(sp.list_readers()))
print(sp.get_reader_info(reader).to_metadata())

raw_segments = [sp.read_spectrum(path, reader=reader) for path in pepsi_paths]
for segment in raw_segments:
    print("", segment.name)
    print(segment.summary())


## 2. Build the legacy line windows

The historical validation compared a small set of PEPSI line windows rather than fitting the whole red spectrum. Spyctres keeps that operation in `sp.recipes.build_pepsi_legacy_segments()` so the notebook does not need to define its own window-building logic.


In [ ]:
input_segments, fit_segments, window_defs_air = sp.recipes.build_pepsi_legacy_segments(
    raw_segments,
    wave_hypothesis=wave_hypothesis,
    halfwidth_A=legacy_halfwidth_A,
    window_pad_A=window_pad_A,
)

collection = sp.SpectrumCollection(
    fit_segments,
    name="example8_pepsi_legacy_line_windows",
    meta={
        "workflow": "example8_pepsi_legacy_linefit_validation",
        "wave_hypothesis": wave_hypothesis,
        "legacy_window_defs_air": [list(item) for item in window_defs_air],
    },
)

summary = collection.summary()
print("Prepared windows:", summary["n_segments"])
print("Total pixels:", summary["n_pixels"])
print("Valid fraction:", f"{summary['valid_fraction']:.3f}")
for item in summary["segments"]:
    print(
        f"{item['name']:<18} {item['n_valid_pixels']:>4}/{item['n_pixels']:<4} "
        f"{item['wavelength_range_A'][0]:.1f}-{item['wavelength_range_A'][1]:.1f} Å"
    )


## 3. Inspect the prepared windows before fitting

Gray points are not currently used by the legacy comparison. This visual check is deliberately separate from the fit: if a PEPSI product has different wavelength or frame conventions, inspect the headers/release notes before changing `wave_hypothesis` or applying velocity corrections.

This panel grid uses the same plotting helper that the optional model fit uses, but here we pass no model arrays. That way the inspection plot and the fitted diagnostic plot have the same visual language.


In [ ]:
fig, axes = sp.plot_prepared_line_window_diagnostics(
    collection,
    title="PEPSI legacy line windows; no PHOENIX fit has been run",
    footer=(
        "Blue = prepared normalized PEPSI windows; gray = pixels not used by "
        "the legacy mask. The orange model overlay appears only in the "
        "optional PHOENIX legacy fit."
    ),
    ncols=min(6, len(collection.segments)),
    figsize_per_panel=(2.25, 2.35),
)
plt.show()


## 4. Optional: run the maintained legacy fit

The full validation fit is intentionally delegated to `scripts/pepsi_fit_smoketest.py`. That script uses the shared PEPSI recipe and PHOENIX backend, prints progress, and is the regression target we keep synchronized with the package.

When this cell runs, the script writes a compact line-window plot to `legacy_fit_plot`. The final notebook cell displays that PNG, so the Figure-4-style diagnostic appears at the end of the worked example.



In [ ]:
legacy_fit_plot.parent.mkdir(parents=True, exist_ok=True)
script_path = ROOT / "scripts" / "pepsi_fit_smoketest.py"
command = [
    sys.executable,
    str(script_path),
    "--preset",
    "pepsi_legacy_red_fast",
    "--wave-hypothesis",
    wave_hypothesis,
    "--output-line-plot",
    str(legacy_fit_plot),
    "--no-show",
    str(pepsi_paths[0]),
    str(pepsi_paths[1]),
]

if RUN_LEGACY_FIT:
    # Avoid accidentally displaying a stale PNG from an older run.
    if legacy_fit_plot.exists():
        legacy_fit_plot.unlink()
    print("Writing PEPSI diagnostic plot to:", legacy_fit_plot)
    subprocess.run(command, check=True)
else:
    print("Legacy PHOENIX fit is disabled. To run it later:")
    print(" ".join(command))



## Interpretation checklist

- The compact plot is a legacy regression diagnostic: inspect whether each line window supports the same model, not just the aggregate fit statistic.
- PEPSI `.nor` products from different releases can have different wavelength conventions.
- For PETS/NASA-style stellar-rest products, do not apply `SSBVEL`/`OBSVEL` again.
- For generic local PEPSI products, verify FITS headers or release documentation before applying velocity corrections.
- Use the reviewed-analysis examples for ordinary PHOENIX interpretation discipline; this example exists mainly to keep the PEPSI path auditable.

The final cell below displays the saved compact diagnostic plot. If it does not appear, check the printed `legacy_fit_plot` path and whether the previous cell completed without error.



In [ ]:
if legacy_fit_plot.exists():
    display(Image(filename=str(legacy_fit_plot)))
    print("Displayed PEPSI legacy diagnostic plot from:", legacy_fit_plot)
else:
    print(
        "No PEPSI legacy fit plot was found. Run the previous cell with "
        "RUN_LEGACY_FIT = True after PHOENIX is configured. Expected path:",
        legacy_fit_plot,
    )

